# Importing the Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score

# Reading the Data

In [ ]:
df = pd.read_csv('spotify dataset.csv')
df['index'] =df.index
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

# Separating the Features

In [ ]:
cat_columns = ['playlist_genre', 'playlist_subgenre', 'playlist_name', 'track_artist', 'track_album_name']

num_columns = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
                  'instrumentalness', 'liveness', 'valence', 'tempo']

In [ ]:
encoders = {}
for col in cat_columns:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])

In [ ]:
scaler = StandardScaler()
df[num_columns] = scaler.fit_transform(df[num_columns])

In [ ]:
df_processed = df[cat_columns + num_columns]

# Data Visualizations

In [ ]:
plt.figure(figsize=(10, 8))
for i, feature in enumerate(num_columns, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[feature], bins=30, kde=True, color='blue')
    plt.title(f'Distribution of {feature}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_processed.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
inertia = []
k_range = range(2, 15)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_processed)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(5, 3))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.show()

In [ ]:
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_processed)

score = silhouette_score(df_processed, df['cluster'])
print(f'Silhouette Score: {score}')

# Cosine Similarity

In [ ]:
similarity = cosine_similarity(df_processed)
print(similarity)
print(similarity.shape)

# Getting the Song from User

In [ ]:
df['track_name'] = df['track_name'].astype(str)

track_name = input('Enter your favourite spotify song: ')

list_of_all_songs = df['track_name'].tolist()

find_close_match = difflib.get_close_matches(track_name, list_of_all_songs)

close_match = find_close_match[0]

index_of_the_song = df[df.track_name == close_match]['index'].values[0]

# Finding the Similarity Score

In [ ]:
similarity_score = list(enumerate(similarity[index_of_the_song]))

sorted_similar_songs = sorted(similarity_score, key = lambda x:x[1], reverse=True)

# Suggesting the Songs

In [ ]:
print("Songs suggested for you : \n")

i = 1

for song in sorted_similar_songs[1:11]:
    index = song[0]
    title_of_the_song = df[df.index == index]['track_name'].values[0]
    if (i<11):
        print(i,'. ', title_of_the_song)
        i += 1